# Dense V1 + S09 rerank trên Colab Pro

Notebook độc lập: dùng 6 ZIP đầu vào Stage 8 để khôi phục structural chunks,
kiểm tra candidates V1 với văn bản nguồn, rerank top 400 rồi xuất 120 docs / 120 chunks.
Không cần BM25, không chạy lại chunking/embedding, không dùng semantic store.
Đây là thí nghiệm dense riêng, không tái hiện baseline hybrid 0.1626.

Chạy từng cell trong runtime mới. Chuẩn bị dùng CPU và ổ đĩa, GPU chỉ dùng ở bước rerank.
Kiểm tra 5 query trước; sau đó đặt STOP_AFTER_QUERIES=None và chạy lại cell worker.
Không tự nộp lên website. Giữ bản nộp tốt nhất và so sánh bằng cùng evaluator mới của BTC.
Luật LCS hai chiều không được đo ở đây: chưa có evaluator/đáp án chính thức.
Notebook giữ nguyên evidence gốc, chưa cắt hoặc viết lại chunk.

Mã nguồn được nhúng sẵn, chỉ cần upload notebook. Các file dữ liệu vẫn phải có trên Drive.
Không chạy hai phiên cùng RUN_DRIVE. Sau khi mất runtime, chạy lại từ đầu:
source được khôi phục lại, candidates được kiểm tra lại, điểm query hoàn tất được lấy từ backup.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, zipfile, tempfile, shutil, sys, subprocess, os

ROOT = Path('/content/drive/MyDrive/ViBioMIR')
BUNDLE = ROOT / 'stage9_rerank_colab.zip'
STAGE8_BUNDLES = ROOT / 'dense_stage8_bge_structure/bundles'
DENSE_VERSION = 'query_results_v1'
DENSE_RESULTS = ROOT / 'dense_stage8_bge_structure' / DENSE_VERSION / 'results'
DATA_ZIP = ROOT / 'Data.zip'
QUERY_FILE = ROOT / 'Data/official/ViBioMIR/query.parquet'
LINKS_FILE = ROOT / 'Data/official/ViBioMIR/links_corpus_vi_en_zh.parquet'
LINKS_ID_COLUMN = 'id'
EXPECTED_LINKS_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
EXPECTED_QUERIES = 1200
LIMITS = dict(max_docs=120, max_chunks=120, max_chunks_per_doc=6, overlap_iou_threshold=0.8)
RERANK_TOP_K = 400
STOP_AFTER_QUERIES = 5
RUN_NAME = f'stage9_dense_{DENSE_VERSION}_k{RERANK_TOP_K}_120_120_v1'
RUN_DRIVE = ROOT / RUN_NAME
LOCAL = Path('/content') / RUN_NAME
CHUNKS = LOCAL / 'chunks'
CANDIDATES = LOCAL / 'validated_candidates'
NEW_PREPARED = LOCAL / 'prepared'
NEW_WORKERS = LOCAL / 'workers'
assert DENSE_VERSION in ('query_results_v1', 'query_results_v2')
assert 0 < RERANK_TOP_K <= 500
assert BUNDLE.is_file(), f'Missing S09 code bundle: {BUNDLE}'
assert STAGE8_BUNDLES.is_dir(), f'Missing Stage-8 input bundles: {STAGE8_BUNDLES}'
assert (DENSE_RESULTS / 'manifest.json').is_file(), f'Missing dense results: {DENSE_RESULTS}'
LOCAL.mkdir(parents=True, exist_ok=True)
RUN_DRIVE.mkdir(parents=True, exist_ok=True)
print('Source bundles:', STAGE8_BUNDLES)
print('Dense results:', DENSE_RESULTS)
print('New output:', RUN_DRIVE)


In [ ]:
# Keep PROJECT creation and extraction in this single cell.
assert not any(n == 'vibiomir' or n.startswith('vibiomir.') for n in sys.modules), 'Restart runtime before rerunning this cell.'
PROJECT = Path(tempfile.mkdtemp(prefix='vibiomir_dense_s09_', dir='/content'))
def digest(data):
    return hashlib.sha256(data).hexdigest()
def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
with zipfile.ZipFile(BUNDLE) as z:
    meta = json.loads(z.read('BUNDLE.json'))
    assert digest(canonical({k:v for k,v in meta.items() if k != 'signature'}).encode()) == meta['signature']
    assert len(z.namelist()) == len(set(z.namelist()))
    assert set(z.namelist()) == set(meta['files']) | {'BUNDLE.json'}
    assert 'vibiomir/stage9_pipeline.py' in meta['files'], 'Wrong bundle: S09 code is missing'
    assert shutil.disk_usage('/content').free > sum(i.file_size for i in z.infolist()) + 1024**3
    for name, expected_hash in meta['files'].items():
        path = (PROJECT / name).resolve()
        assert PROJECT.resolve() in path.parents
        data = z.read(name)
        assert digest(data) == expected_hash, f'Corrupt bundle member: {name}'
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
assert (PROJECT / 'vibiomir/stage9_pipeline.py').is_file()
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)  # Frozen bundle() reads requirements-stage9-colab.txt relative to cwd.
import importlib
importlib.invalidate_caches()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow==22.0.0'], check=True)
from vibiomir.stage9_pipeline import prepared, prepare, bundle, export, validate_results, validate_worker
from vibiomir.retrieval_contracts import queries, chunk_snapshot
from vibiomir.io import file_hash, write_json
state = prepared(PROJECT)
assert state['identity']['scope'] == 'all_queries' and len(state['packets']) == EXPECTED_QUERIES
expected_source = state['identity']['source']
print('Loaded frozen S09 code:', PROJECT)
print('Expected structural source:', expected_source)


In [ ]:
import pyarrow.parquet as pq
def official_file(path, basenames):
    if path.is_file():
        return path
    assert DATA_ZIP.is_file(), f'Missing {path}. Set the path or provide Data.zip.'
    with zipfile.ZipFile(DATA_ZIP) as z:
        matches = [n for n in z.namelist() if Path(n).name in basenames and 'official' in Path(n).parts]
        assert len(matches) == 1, f'Expected one official file, found {matches}. Extract the intended file and set its path.'
        target = PROJECT / 'official' / Path(matches[0]).name
        target.parent.mkdir(exist_ok=True)
        with z.open(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        return target
query_path = official_file(QUERY_FILE, {'query.parquet'})
links_path = official_file(LINKS_FILE, {'links_corpus_vi_en_zh.parquet', 'links_corpus.parquet'})
official_queries = queries(query_path)
assert file_hash(query_path) == state['identity']['query_file_sha256'], 'Query snapshot differs from frozen run.'
assert [q['id'] for q in official_queries] == [p['id'] for p in state['packets']]
assert file_hash(links_path) == EXPECTED_LINKS_SHA256, 'Official links differ from the verified baseline snapshot; verify the file before continuing.'
pf = pq.ParquetFile(links_path)
assert LINKS_ID_COLUMN in pf.schema_arrow.names, f'Set LINKS_ID_COLUMN. Available: {pf.schema_arrow.names}'
allowed = set()
for batch in pf.iter_batches(columns=[LINKS_ID_COLUMN]):
    for value in batch.column(0).to_pylist():
        assert type(value) is int, 'Official IDs must be integers.'
        allowed.add(value)
assert allowed


In [ ]:
RECOVERY_SOURCE = '"""Recover Stage-8 source bundles and validate dense candidates for frozen S09."""\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport sqlite3\nimport zipfile\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'))\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(canonical(value), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef signed(value):\n    return {**value, \'signature\': digest(canonical(value).encode())}\n\n\ndef checked(value):\n    require(value.get(\'signature\') == signed({k: v for k, v in value.items() if k != \'signature\'})[\'signature\'],\n            \'Signed record checksum differs\')\n    return value\n\n\ndef inspect_bundles(directory, expected):\n    archives = sorted(Path(directory).glob(\'worker_*.zip\'))\n    require(archives, f\'No Stage-8 input bundles in {directory}\')\n    plan, members, metadata = None, {}, {}\n    for archive in archives:\n        with zipfile.ZipFile(archive) as z:\n            names = z.namelist()\n            require(len(names) == len(set(names)), f\'Duplicate ZIP members: {archive}\')\n            current = checked(json.loads(z.read(\'plan.json\')))\n            require(current[\'source\'] == expected, f\'Stage-8 source differs: {archive}\')\n            require(plan is None or current == plan, \'Mixed Stage-8 plans\')\n            plan = current\n            for name, key in [(\'manifest.json\', \'manifest_sha256\'), (\'VALIDATION.json\', \'validation_sha256\')]:\n                data = z.read(\'chunks/\' + name)\n                require(digest(data) == expected[key], f\'Source metadata differs: {archive}/{name}\')\n                metadata[name] = data\n            for part in plan[\'parts\']:\n                index = part[\'index\']\n                prefix = f\'chunks/batch-{index:06d}/\'\n                if prefix + \'documents.jsonl.gz\' not in names:\n                    continue\n                require(prefix + \'batch.json\' in names, f\'Missing batch metadata: {prefix}\')\n                require(index not in members, f\'Duplicate batch across bundles: {index}\')\n                require(digest(z.read(prefix + \'batch.json\')) == part[\'batch_json_sha256\'], \'Batch metadata differs\')\n                members[index] = (archive, prefix, z.getinfo(prefix + \'documents.jsonl.gz\').file_size)\n        print(f\'Inspected {archive.name}; total unique batches: {len(members)}\', flush=True)\n    require(set(members) == set(range(expected[\'batches\'])), \'Missing or unexpected source batches\')\n    require(len(plan[\'parts\']) == expected[\'batches\'], \'Plan batch count differs\')\n    require(sum(p[\'rows\'] for p in plan[\'parts\']) == expected[\'chunks\'] and\n            sum(p[\'documents\'] for p in plan[\'parts\']) == expected[\'documents\'], \'Plan totals differ\')\n    return plan, members, metadata\n\n\ndef restore_chunks(directory, destination, expected):\n    plan, members, metadata = inspect_bundles(directory, expected)\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    for name, data in metadata.items():\n        if (destination / name).exists():\n            require((destination / name).read_bytes() == data, \'Destination contains another source; use a new folder\')\n    missing_bytes = sum(size for index, (_, _, size) in members.items()\n                        if not (destination / f\'batch-{index:06d}/documents.jsonl.gz\').is_file())\n    require(shutil.disk_usage(destination).free > missing_bytes + 8 * 1024**3,\n            \'Insufficient local disk for source plus 8 GiB reserve\')\n    # Open each ZIP once. All output names come from integer indices, never arbitrary ZIP paths.\n    done = 0\n    for archive in sorted({v[0] for v in members.values()}):\n        with zipfile.ZipFile(archive) as z:\n            for index, (owner, prefix, _) in sorted(members.items()):\n                if owner != archive:\n                    continue\n                part = plan[\'parts\'][index]\n                require(part[\'index\'] == index, \'Plan parts are not indexed in order\')\n                target = destination / f\'batch-{index:06d}\'\n                target.mkdir(exist_ok=True)\n                for name, expected_hash in [(\'batch.json\', part[\'batch_json_sha256\']),\n                                            (\'documents.jsonl.gz\', part[\'source_sha256\'])]:\n                    path = target / name\n                    if path.is_file() and file_hash(path) == expected_hash:\n                        continue\n                    temporary = path.with_suffix(path.suffix + \'.partial\')\n                    with z.open(prefix + name) as src, temporary.open(\'wb\') as dst:\n                        shutil.copyfileobj(src, dst, length=4 * 1024**2)\n                    require(file_hash(temporary) == expected_hash, f\'Corrupted payload: {prefix}{name}\')\n                    temporary.replace(path)\n                done += 1\n                if done == 1 or done % 25 == 0 or done == len(members):\n                    print(f\'Restored and verified {done}/{len(members)} batches\', flush=True)\n    require({p.name for p in destination.glob(\'batch-*\')} == {f\'batch-{i:06d}\' for i in members},\n            \'Unexpected destination batches\')\n    for name, data in metadata.items():\n        (destination / name).write_bytes(data)\n    return plan\n\n\nFIELDS = (\'chunk_id\', \'doc_id\', \'derived_version_id\', \'source_batch\', \'source_row\',\n          \'chunk_order\', \'retrieval_text_sha256\')\n\n\ndef adapt_candidates(results, chunks, plan, expected, query_path, queries, output):\n    """Validate every candidate against original text before issuing an adapter report."""\n    results, chunks, output = Path(results), Path(chunks), Path(output)\n    manifest = read(results / \'manifest.json\')\n    ident = manifest[\'identity\']\n    require(manifest[\'signature\'] == digest(canonical(ident).encode()) and manifest[\'state\'] == \'complete\',\n            \'Dense result manifest identity/state differs\')\n    require(ident[\'operation\'] in (\'dense-results\', \'dense-v2-cached-candidates\'), \'Unsupported candidate producer\')\n    require(ident[\'plan_signature\'] == plan[\'signature\'] and plan[\'source\'] == expected, \'Candidate plan/source differs\')\n    require(ident[\'query_sha256\'] == file_hash(query_path), \'Candidate query snapshot differs\')\n    require(manifest.get(\'corpus_complete\') is True, \'Candidate producer reports incomplete corpus\')\n    input_hashes = {str(q[\'id\']): file_hash(results / \'candidates\' / f"query-{q[\'id\']}.json") for q in queries}\n    identity = dict(operation=\'dense-candidate-adapter-v1\', source=expected,\n                    input_manifest_sha256=file_hash(results / \'manifest.json\'), query_sha256=file_hash(query_path),\n                    input_query_hashes=input_hashes, adapter_sha256=file_hash(__file__))\n    signature = digest(canonical(identity).encode())\n    output.mkdir(parents=True, exist_ok=True)\n    if (output / \'manifest.json\').exists():\n        require(read(output / \'manifest.json\')[\'identity\'] == identity, \'Adapter inputs changed; use a new output folder\')\n    write(output / \'manifest.json\', dict(identity=identity, signature=signature, state=\'building\'))\n    # This index is disposable; rebuilt on every validation attempt to avoid stale verification flags.\n    with sqlite3.connect(output / \'verification.sqlite3\') as db:\n        db.execute(\'DROP TABLE IF EXISTS wanted\')\n        db.execute(\'CREATE TABLE wanted(id TEXT PRIMARY KEY,batch INTEGER,payload TEXT,verified INTEGER DEFAULT 0)\')\n        total = 0\n        for number, q in enumerate(queries, 1):\n            path = results / \'candidates\' / f"query-{q[\'id\']}.json"\n            raw = path.read_bytes()\n            require(digest(raw) == input_hashes[str(q[\'id\'])], \'Candidate file changed while reading\')\n            record = checked(json.loads(raw))\n            require(record[\'id\'] == q[\'id\'] and record[\'query\'] == q[\'query\'] and\n                    record[\'run_signature\'] == manifest[\'signature\'], \'Candidate query/run binding differs\')\n            seen, previous = set(), math.inf\n            for item in record[\'candidates\']:\n                require(all(k in item for k in FIELDS), \'Missing candidate mapping\')\n                require(isinstance(item[\'chunk_id\'], str) and item[\'chunk_id\'] and type(item[\'doc_id\']) is int,\n                        \'Invalid candidate ID\')\n                require(all(type(item[k]) is int and item[k] >= 0 for k in (\'source_batch\', \'source_row\', \'chunk_order\')),\n                        \'Invalid source coordinates\')\n                require(item[\'source_batch\'] < expected[\'batches\'], \'Unknown batch\')\n                score = item[\'score\']\n                require(type(score) in (int, float) and math.isfinite(score) and score <= previous,\n                        \'Invalid or unsorted candidate scores\')\n                require(item[\'chunk_id\'] not in seen, \'Duplicate candidate in query\')\n                seen.add(item[\'chunk_id\']); previous = score\n                payload = canonical({k: item[k] for k in FIELDS})\n                old = db.execute(\'SELECT payload FROM wanted WHERE id=?\', (item[\'chunk_id\'],)).fetchone()\n                require(old is None or old[0] == payload, \'Conflicting candidate mapping\')\n                db.execute(\'INSERT OR IGNORE INTO wanted(id,batch,payload) VALUES(?,?,?)\',\n                           (item[\'chunk_id\'], item[\'source_batch\'], payload))\n            total += len(record[\'candidates\'])\n            write(output / f"query-{q[\'id\']}.json", signed(dict(id=q[\'id\'], query=q[\'query\'],\n                  run_signature=signature, candidates=record[\'candidates\'])))\n            if number % 100 == 0:\n                print(f\'Checked candidate records: {number}/{len(queries)}\', flush=True)\n        db.commit()\n        db.execute(\'CREATE INDEX wanted_batch ON wanted(batch)\')\n        for part in plan[\'parts\']:\n            index = part[\'index\']\n            wanted = dict(db.execute(\'SELECT id,payload FROM wanted WHERE batch=?\', (index,)))\n            if not wanted:\n                continue\n            path = chunks / f\'batch-{index:06d}/documents.jsonl.gz\'\n            require(file_hash(path) == part[\'source_sha256\'], \'Restored source payload changed\')\n            matched = set()\n            with gzip.open(path, \'rt\', encoding=\'utf-8\') as stream:\n                for row_number, line in enumerate(stream):\n                    document = json.loads(line)\n                    for order, chunk in enumerate(document[\'chunks\']):\n                        key = chunk[\'chunk_id\']\n                        if key not in wanted:\n                            continue\n                        require(key not in matched, \'Duplicate source chunk ID\')\n                        actual = {**chunk, \'source_batch\': index, \'source_row\': row_number, \'chunk_order\': order,\n                                  \'retrieval_text_sha256\': digest(chunk[\'retrieval_text\'].encode())}\n                        require(canonical({k: actual[k] for k in FIELDS}) == wanted[key], \'Candidate does not match source text\')\n                        require(chunk[\'doc_id\'] == document[\'doc_id\'] and\n                                chunk[\'derived_version_id\'] == document[\'derived_version_id\'] and\n                                digest(chunk[\'chunk_text\'].encode()) == chunk[\'text_sha256\'], \'Source evidence differs\')\n                        matched.add(key)\n            require(matched == set(wanted), f\'Missing evidence in batch {index}\')\n            db.execute(\'UPDATE wanted SET verified=1 WHERE batch=?\', (index,))\n            db.commit()\n            if index % 25 == 0:\n                print(f\'Verified source mappings through batch {index}/{expected["batches"]}\', flush=True)\n        require(db.execute(\'SELECT COUNT(*) FROM wanted WHERE verified=0\').fetchone()[0] == 0, \'Unverified candidates\')\n    write(output / \'manifest.json\', dict(identity=identity, signature=signature, state=\'complete\',\n                                        queries=len(queries), candidates=total))\n    report = dict(validation=\'passed\', validation_scope=\'candidate integrity and exact source mapping; not relevance\',\n                  manifest_sha256=file_hash(output / \'manifest.json\'), query_sha256=file_hash(query_path),\n                  queries=len(queries), candidates=total, relevance_measured=False)\n    write(output / \'VALIDATION.json\', report)\n    return report\n'
PACKAGE_SOURCE = '"""Verified score reuse and submission packaging for an expanded S09 pool."""\nimport json\nfrom pathlib import Path\nimport sqlite3\nimport zipfile\n\nfrom vibiomir.io import canonical_json, file_hash, text_hash, write_json, writer_lock\nfrom vibiomir.retrieval_contracts import checked, open_run, require\nfrom vibiomir.stage9_core import score_key\nfrom vibiomir.stage9_pipeline import prepared, load_packet, verify_scores, _worker_identity\n\n\ndef seed_cache(old_dir, old_workers, new_dir, output, worker_id=0):\n    """Import verified exact query/passage/policy scores; never reuse chunk-ID alone."""\n    old, new = prepared(old_dir), prepared(new_dir)\n    policy = new[\'identity\'][\'settings\'][\'reranker\']\n    require(old[\'identity\'][\'settings\'][\'reranker\'] == policy, \'Reranker policy differs\')\n    require(old[\'identity\'][\'source\'] == new[\'identity\'][\'source\'], \'Corpus snapshot differs\')\n    require(old[\'identity\'][\'code\'] == new[\'identity\'][\'code\'], \'Frozen scoring code differs\')\n    require(old[\'identity\'][\'query_file_sha256\'] == new[\'identity\'][\'query_file_sha256\'], \'Query snapshot differs\')\n    old_entries = {p[\'id\']: p for p in old[\'packets\']}\n    root = Path(output)\n    identity = _worker_identity(new, worker_id)\n    open_run(root, identity)\n    reused, missing, files = 0, 0, {}\n    with writer_lock(root), sqlite3.connect(root / \'score_cache.sqlite3\') as db:\n        db.execute(\'CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,payload TEXT NOT NULL)\')\n        for entry in new[\'packets\']:\n            if entry[\'worker_id\'] != worker_id:\n                continue\n            packet = load_packet(new_dir, entry)\n            previous = old_entries.get(entry[\'id\'])\n            require(previous is not None, \'Expanded run has an unknown query\')\n            original = load_packet(old_dir, previous)\n            require(original[\'query\'] == packet[\'query\'], \'Query text changed\')\n            old_worker = Path(old_workers) / f"worker_{previous[\'worker_id\']}"\n            manifest = json.loads((old_worker / \'manifest.json\').read_text(encoding=\'utf-8\'))\n            expected = _worker_identity(old, previous[\'worker_id\'])\n            require(manifest[\'identity\'] == expected and manifest[\'signature\'] == text_hash(canonical_json(expected))\n                    and manifest[\'state\'] == \'complete\', \'Old worker not complete or mismatched\')\n            path = old_worker / f"query-{previous[\'id\']}.json"\n            rows = verify_scores(checked(path), original, previous, old, manifest[\'signature\'])\n            files[str(previous[\'id\'])] = file_hash(path)\n            scores = {}\n            for item, value in zip(original[\'candidates\'], rows):\n                key = score_key(original[\'query\'], item[\'retrieval_text\'], policy)\n                payload = {k: v for k, v in value.items() if k != \'chunk_id\'}\n                require(key not in scores or scores[key] == payload, \'Conflicting old scores for identical text\')\n                scores[key] = payload\n            for item in packet[\'candidates\']:\n                key = score_key(packet[\'query\'], item[\'retrieval_text\'], policy)\n                if key not in scores:\n                    missing += 1\n                    continue\n                payload = canonical_json(scores[key])\n                hit = db.execute(\'SELECT payload FROM scores WHERE key=?\', (key,)).fetchone()\n                require(hit is None or json.loads(hit[0]) == scores[key], \'Existing cache conflicts with verified score\')\n                db.execute(\'INSERT OR IGNORE INTO scores VALUES(?,?)\', (key, payload))\n                reused += 1\n            db.commit()\n    report = dict(old_signature=old[\'signature\'], new_signature=new[\'signature\'],\n                  reused_pairs=reused, new_pairs=missing, old_score_files=files,\n                  helper_sha256=file_hash(__file__))\n    write_json(root / \'REUSE.json\', report)\n    return report\n\n\ndef package(results, query_ids, allowed_ids, destination):\n    """Package only source-validated results with exact query and ID coverage."""\n    results, destination = Path(results), Path(destination)\n    report = json.loads((results / \'VALIDATION.json\').read_text(encoding=\'utf-8\'))\n    require(report[\'validation\'] == \'passed\' and report[\'original_evidence_verified\'] is True,\n            \'Original source validation is required\')\n    require(file_hash(results / \'manifest.json\') == report[\'manifest_sha256\'], \'Result manifest changed\')\n    path = results / \'predictions.json\'\n    require(file_hash(path) == report[\'predictions_sha256\'], \'Prediction hash differs\')\n    payload = path.read_bytes()\n    rows = json.loads(payload)\n    require([r[\'id\'] for r in rows] == query_ids and len(set(query_ids)) == len(query_ids), \'Query coverage differs\')\n    for row in rows:\n        docs = row[\'relevant_docs\']\n        require(all(type(d) is int and d in allowed_ids for d in docs) and len(docs) == len(set(docs)), \'Invalid document IDs\')\n        for chunk in row[\'relevant_chunks\']:\n            require(type(chunk[\'doc_id\']) is int and chunk[\'doc_id\'] in docs and\n                    isinstance(chunk[\'chunk_text\'], str) and chunk[\'chunk_text\'].strip(), \'Invalid evidence\')\n    destination.mkdir(parents=True, exist_ok=False)\n    (destination / \'submission.json\').write_bytes(payload)\n    archive = destination / \'submission.zip\'\n    with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED) as z:\n        z.writestr(\'submission.json\', payload)\n    with zipfile.ZipFile(archive) as z:\n        require(z.namelist() == [\'submission.json\'] and z.read(\'submission.json\') == payload, \'ZIP differs\')\n    write_json(destination / \'REPORT.json\', dict(queries=len(rows), source_validation=report,\n               submission_sha256=file_hash(path), zip_sha256=file_hash(archive), relevance_measured=False))\n    return archive\n'
(PROJECT / 'dense_recovery_support.py').write_text(RECOVERY_SOURCE, encoding='utf-8')
(PROJECT / 'dense_package_support.py').write_text(PACKAGE_SOURCE, encoding='utf-8')
from dense_recovery_support import inspect_bundles, restore_chunks, adapt_candidates
from dense_package_support import package
import time, threading
from contextlib import contextmanager
@contextmanager
def progress(label):
    started = time.monotonic()
    event = threading.Event()
    def heartbeat():
        while not event.wait(30):
            print(label, 'running minutes:', round((time.monotonic()-started)/60, 1), flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    print(label, flush=True)
    thread.start()
    try:
        yield
    finally:
        event.set()
        thread.join()


In [ ]:
# Read metadata before restoring large payloads.
with progress('Inspect six Stage-8 bundles'):
    plan, members, metadata = inspect_bundles(STAGE8_BUNDLES, expected_source)
dense_manifest = json.loads((DENSE_RESULTS / 'manifest.json').read_text())
assert dense_manifest['state'] == 'complete' and dense_manifest.get('corpus_complete') is True
assert dense_manifest['identity']['plan_signature'] == plan['signature']
assert dense_manifest['identity']['query_sha256'] == file_hash(query_path)
assert dense_manifest['signature'] == digest(canonical(dense_manifest['identity']).encode())
missing = [q['id'] for q in official_queries if not (DENSE_RESULTS / 'candidates' / f"query-{q['id']}.json").is_file()]
assert not missing, f'Missing candidate queries: {missing[:20]}'
required_gib = sum(v[2] for v in members.values()) / 1024**3
print('Source compressed payload size after ZIP extraction, GiB:', round(required_gib, 2))
print('Free local GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))
print('Metadata checks passed; payload validation follows.')


In [ ]:
with progress('Restore structural source; completed files are checksum-verified and reused'):
    plan = restore_chunks(STAGE8_BUNDLES, CHUNKS, expected_source)
assert chunk_snapshot(CHUNKS) == expected_source
print('Structural store restored:', CHUNKS)


In [ ]:
# This can take time: validate ALL candidates, not only a sample.
with progress('Validate dense candidate records and source text on CPU'):
    report = adapt_candidates(DENSE_RESULTS, CHUNKS, plan, expected_source,
                              query_path, official_queries, CANDIDATES)
write_json(RUN_DRIVE / 'CANDIDATE_VALIDATION.json', report)
print(report)


In [ ]:
fixed_query = LOCAL / 'query.parquet'
shutil.copy2(query_path, fixed_query)
config = json.loads(json.dumps(state['identity']['settings']))
config.update(queries=str(fixed_query), chunk_dir=str(CHUNKS), output_dir=str(NEW_PREPARED),
              rerank_top_k=RERANK_TOP_K, fused_top_k=500, fusion='rrf', workers=1,
              selection=LIMITS,
              branches=[dict(name='dense_verified', family='dense', facet_id='original',
                             candidate_dir=str(CANDIDATES))])
write_json(LOCAL / 'config.json', config)
with progress('Prepare S09 packets from verified dense candidates'):
    checkpoint = RUN_DRIVE / 'prepared.zip'
    if checkpoint.is_file():
        # Restore the exact gzip bytes referenced by completed worker scores.
        with zipfile.ZipFile(checkpoint) as z:
            meta = json.loads(z.read('BUNDLE.json'))
            assert digest(canonical({k:v for k,v in meta.items() if k != 'signature'}).encode()) == meta['signature']
            assert len(z.namelist()) == len(set(z.namelist()))
            assert set(z.namelist()) == set(meta['files']) | {'BUNDLE.json'}
            for name, expected_hash in meta['files'].items():
                target = (NEW_PREPARED / name).resolve()
                assert NEW_PREPARED.resolve() in target.parents
                payload = z.read(name)
                assert digest(payload) == expected_hash
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(payload)
    expanded = prepare(LOCAL / 'config.json')
assert len(expanded['packets']) == EXPECTED_QUERIES
assert expanded['identity']['source'] == expected_source
with progress('Back up prepared bundle'):
    local_zip = LOCAL / 'prepared.zip'
    bundle(NEW_PREPARED, local_zip)
    pending = RUN_DRIVE / 'prepared.zip.tmp'
    shutil.copy2(local_zip, pending)
    assert file_hash(pending) == file_hash(local_zip)
    pending.replace(RUN_DRIVE / 'prepared.zip')
print('Prepared pairs:', expanded['candidates'])
print('No baseline hybrid scores have been imported; new dense run.')


In [ ]:
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-stage9-colab.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime.'
from vibiomir.stage9_reranker import recommended_runtime
preset = recommended_runtime(torch.cuda.get_device_properties(0).total_memory / 1024**3)
print(torch.cuda.get_device_name(0), preset)


In [ ]:
# Separate checkpoints by exact packet bytes, not only logical run identity.
packet_state = prepared(NEW_PREPARED)
packet_tag = digest(canonical([(p['id'], p['sha256']) for p in packet_state['packets']]).encode())
NEW_WORKERS = LOCAL / 'workers_by_packets' / packet_tag
WORKER_BACKUP = RUN_DRIVE / 'workers_by_packets' / packet_tag / 'worker_0'
print('Packet checkpoint:', packet_tag, flush=True)
print('Worker backup:', WORKER_BACKUP, flush=True)
# Rerun this cell with STOP_AFTER_QUERIES=None after checking pilot time.
import time
command = [sys.executable, '-u', '-m', 'vibiomir.stage9_pipeline', 'rerank-worker',
           '--prepared-dir', str(NEW_PREPARED), '--output-dir', str(NEW_WORKERS / 'worker_0'),
           '--worker-id', '0', '--backup-dir', str(WORKER_BACKUP),
           '--batch-size', str(preset['batch_size']), '--padded-tokens', str(preset['padded_tokens'])]
if STOP_AFTER_QUERIES is not None:
    command += ['--stop-after-queries', str(STOP_AFTER_QUERIES)]
started = time.monotonic()
log = RUN_DRIVE / 'latest_worker.log'
with log.open('w', encoding='utf-8') as stream:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            stream.write(line); stream.flush()
        status = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
assert status == 0, f'Worker failed; inspect {log}. Completed query backups are retained.'
result = json.loads((NEW_WORKERS / 'worker_0/manifest.json').read_text())
print('Elapsed seconds:', round(time.monotonic()-started, 1), 'State:', result['state'], 'Queries:', result['queries'])
if result['state'] != 'complete':
    print('Pilot/partial run: set STOP_AFTER_QUERIES=None and rerun this GPU cell. Do not export yet.')


In [ ]:
worker = NEW_WORKERS / 'worker_0'
assert json.loads((worker / 'manifest.json').read_text())['state'] == 'complete', 'Finish all queries before export'
with progress('Validate worker and original evidence; export submission'):
    validate_worker(NEW_PREPARED, worker, 0)
    RESULTS = LOCAL / 'results'
    export(NEW_PREPARED, RESULTS, mode='reranked', workers_dir=NEW_WORKERS)
    validation = validate_results(NEW_PREPARED, RESULTS, chunk_dir=CHUNKS,
                                  queries_path=fixed_query, workers_dir=NEW_WORKERS)
    from datetime import datetime, timezone
    destination = RUN_DRIVE / ('submission_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    archive = package(RESULTS, [q['id'] for q in official_queries], allowed, destination)
    shutil.copy2(RESULTS / 'VALIDATION.json', destination / 'VALIDATION.json')
    write_json(destination / 'EXPERIMENT.json', dict(kind='dense-only-cross-encoder',
        dense_results=str(DENSE_RESULTS), source=expected_source, limits=LIMITS,
        rerank_top_k=RERANK_TOP_K, relevance_measured=False,
        official_lcs_evaluator_run=False, chunk_text_policy='original source chunks unchanged'))
print('Submission ZIP:', archive)
print('Technical validation passed. Relevance/LCS score has not been measured.')
from google.colab import files
files.download(str(archive))
